# Qwen and vLLM: preparation · 2.1.9

Declaring a model artifact or server does not start a GPU process or download weights. [Prepare WSL](../../docs/VLLM_GPU_214.md). A missing endpoint is reported as pending.

## Validated 4B deployment configurations

A separate Agentic Systems 2.1.9 campaign passed 22/22 checks on both Colab L4 and SageMaker L4 using `unsloth/Qwen3-4B-Instruct-2507` and `unsloth/Qwen3-4B-Thinking-2507`, loaded sequentially. See the [configuration and evidence report](../../docs/VALIDATION_215_20261108.md).

That campaign uses explicit server settings, not the named fast/medium/power presets. It does not execute or certify the Qwen3-0.6B example below. The example remains a declaration and an opt-in live cell requiring its own matching server.


In [ ]:
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.9'


In [ ]:
artifact = toolkit.model_artifact('Qwen/Qwen3-0.6B', revision='main', metadata={'scope': 'declaration only'})
server = toolkit.model_server(artifact, max_model_len=2048, max_num_seqs=1, gpu_memory_utilization=0.75)
print(artifact.inspect())
print(server.inspect())


In [ ]:
@toolkit.tool
def certify(value: int) -> dict:
    return {'value': value, 'verified': True}
capability = toolkit.skill(name='evidence', tools=(certify,))
actor_framework = 'native'
runtime = toolkit.runtime(provider='vllm-runtime', model='Qwen/Qwen3-0.6B', endpoint='http://127.0.0.1:8000/v1', api_key='vllm', max_tokens=512, max_retries=0)
actor = toolkit.Agent('verification', runtime=runtime, framework=toolkit.framework(actor_framework), skills=(capability,), instructions='Call certify once with the supplied value. The Tool output is the result.', contract={'tool_expectation': {'all_of': ['certify']}, 'completion': 'when_required_tools_satisfied'}, policy={'repair': False})
result = actor.run({'value': 37}, timeout_s=120)
events = [event for node in result.walk() for event in node.tool_events]
assert result.ok and result.check_invariants().ok, result.evidence()
assert any((e.name == 'certify' and e.ok and (e.input == {'value': 37}) for e in events))
print({'input': {'value': 37}, 'process': 'runtime + actor -> Tool event', 'output': result.data})
